<a href="https://colab.research.google.com/github/informraajesh/outskill-ai-lab/blob/main/1_LangChain_end_to_end.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LangChain End-to-End Demo [v1 (Oct 2025) — Updated ]


# LangChain End-to-End: Prompt → Chain → RAG → Agent → UI

This single notebook merges the **basics** and **agent** demos into one coherent flow:

1. **Hello LLM** (baseline)  
2. **Prompting + LCEL + Output Parser**  
3. **RAG (build once, re-use)** with sources  
4. **Agent + Tools** (retriever tool + optional web search)  
5. **Tiny UI** (Gradio)  
6. **Custom Tool (@tool) example**  

> **Tip:** Open the notebook command palette and run all cells, or step through section by section.

## 0) Install & Configure (run once)

- Installs pinned versions to reduce API drift.  
- Prompts for your keys as needed.  
- **Optional**: If you have a LangSmith key, tracing will be enabled automatically.

> If you re-run the notebook later, you can skip re-installation if your environment already has these packages.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
def ignore_warnings(*args, **kwargs):
    pass
warnings.showwarning = ignore_warnings


In [ ]:
# Install LangChain & LangGraph v1 with compatible integrations
%pip install -qU \
    "requests>=2.32.5" \
    "langchain>=1.0.3,<2" \
    "langgraph>=1.1.5,<2" \
    "langgraph-prebuilt>=1.0.9" \
    "langchain-openai>=1.0" \
    "langchain-community>=0.4,<1.0" \
    "langchain-text-splitters>=1.0.0" \
    beautifulsoup4 lxml faiss-cpu langchainhub tavily-python "gradio>=4.0"

!pip show langchain langgraph langgraph-prebuilt langchain-openai | grep -E "Name|Version"
from langgraph.runtime import ExecutionInfo, ServerInfo
from langchain_openai import ChatOpenAI
print("OK")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.1/113.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.8/173.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.7/19.7 MB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 k

In [ ]:
import importlib
def _ver(name):
    try:
        m = importlib.import_module(name)
        return getattr(m, "__version__", "n/a")
    except Exception as e:
        return f"not installed ({e})"
print("langchain           :", _ver("langchain"))
print("langgraph           :", _ver("langgraph"))
print("langchain-core      :", _ver("langchain_core"))
print("langchain-community :", _ver("langchain_community"))
print("langchain-openai    :", _ver("langchain_openai"))
print("langchainhub        :", _ver("langchainhub"))
print("langchain-text-splitters:", _ver("langchain_text_splitters"))
print("faiss-cpu           :", _ver("faiss"))
print("tavily-python       :", _ver("tavily"))


langchain           : 1.2.17
langgraph           : n/a
langchain-core      : 1.3.2
langchain-community : 0.4.1
langchain-openai    : n/a
langchainhub        : n/a
langchain-text-splitters: n/a
faiss-cpu           : 1.13.2
tavily-python       : n/a


In [ ]:
from google.colab import userdata
TAVILY_API_KEY = userdata.get('TAVILY_API_KEY')
LANGCHAIN_API_KEY = userdata.get('LANGCHAIN_API_KEY')
OPEN_AI_KEY = userdata.get('OPEN_AI_KEY_NEW')
print(f"""
TAVILY_API_KEY = {TAVILY_API_KEY[:5]}...
LANGCHAIN_API_KEY = {LANGCHAIN_API_KEY[:5]}...
OPEN_AI_KEY = {OPEN_AI_KEY[:5]}...
""")


TAVILY_API_KEY = tvly-...
LANGCHAIN_API_KEY = lsv2_...
OPEN_AI_KEY = sk-pr...



In [ ]:
import os

os.environ["LANGCHAIN_API_KEY"] = LANGCHAIN_API_KEY
os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY
os.environ["OPENAI_API_KEY"] = OPEN_AI_KEY

In [ ]:
import os
from typing import List, Any
from langchain.agents import create_agent
from langchain_core.tools import create_retriever_tool
try:
    from langchain_community.tools.tavily_search import TavilySearchResults
except Exception:
    TavilySearchResults = None

DEFAULT_MODEL = os.getenv("LC_V1_MODEL", "gpt-4o-mini")

def build_v1_agent(tools: List[Any], system_prompt: str = "You are a helpful assistant."):
    # In v1, `model` can be a string model id (e.g., 'gpt-4o-mini') or a chat model instance.
    return create_agent(model=DEFAULT_MODEL, tools=tools, system_prompt=system_prompt)

def run_agent(agent, question: str):
    try:
        return agent.invoke({"messages": [{"role": "user", "content": question}]})
    except Exception:
        return agent.invoke(question)


In [ ]:

import os, getpass, warnings, sys

warnings.filterwarnings("ignore")

def ensure_env(key: str, prompt: str):
    if not os.getenv(key):
        try:
            val = getpass.getpass(prompt)
        except Exception:
            # Fallback for environments without stdin (e.g. some hosted notebooks)
            val = ""
        if val:
            os.environ[key] = val

# --- Required for LLM & embeddings ---
ensure_env("OPENAI_API_KEY", "Enter your OpenAI API Key (skipped if already set): ")

# --- Optional: LangSmith tracing ---
# If you have LANGCHAIN_API_KEY set, we turn on tracing v2 automatically.
if os.getenv("LANGCHAIN_API_KEY"):
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    print("LangSmith tracing enabled (TRACING_V2=true).")
else:
    os.environ.pop("LANGCHAIN_TRACING_V2", None)
    print("LangSmith tracing not enabled (no LANGCHAIN_API_KEY). Proceeding without tracing.")

# --- Optional: Tavily web search ---
# If not set, we'll skip adding the Tavily tool; everything else runs fine.
if not os.getenv("TAVILY_API_KEY"):
    print("No TAVILY_API_KEY found. Agent will run without web search tool (retriever-only).")
else:
    print("Tavily web search tool will be available.")


LangSmith tracing enabled (TRACING_V2=true).
Tavily web search tool will be available.


## 1) Hello LLM (baseline)

A single LLM call; no structure, no grounding.  
We'll improve over this baseline throughout the notebook.

In [ ]:

from langchain_openai import ChatOpenAI

# A deterministic model (temperature=0) for reproducible outputs.
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

baseline_q = "In one sentence, how can LangSmith help with testing LLM apps?"
baseline_a = llm.invoke(baseline_q)
print("Q:", baseline_q)
print("\nBaseline (no context):\n", baseline_a.content)


Q: In one sentence, how can LangSmith help with testing LLM apps?

Baseline (no context):
 LangSmith can provide comprehensive testing services for LLM apps to ensure they meet quality standards and perform optimally.


## 2) Prompting + LCEL + Output Parsing

Use **LCEL(LangChain Expression Language)** (`|`) to pipe **PromptTemplate → LLM → OutputParser** so your code is composable and testable.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate   # ✅ v1 path
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# LLM (uses OPENAI_API_KEY from env)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Prompt → LLM → Parser, piped with LCEL `|`
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise technical assistant."),
    ("human", "{question}")
])

chain = prompt | llm | StrOutputParser()

# Try it
chain.invoke({"question": "In one sentence, how can LangSmith help with testing LLM apps?"})


'LangSmith provides tools for creating, managing, and automating tests for LLM applications, enabling developers to ensure the accuracy and reliability of their models through structured testing frameworks.'

In [ ]:
llm.invoke('In one sentence, how can LangSmith help with testing LLM apps?')

AIMessage(content='LangSmith provides a framework for testing LLM applications by enabling developers to create, manage, and execute test cases that ensure the reliability and performance of their language models.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 22, 'total_tokens': 55, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_57a1c65a0e', 'id': 'chatcmpl-Db44MAbnzL6FzgVXV7943OMMux9DO', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019de8bc-51e6-7753-8540-8b4e8519334e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 22, 'output_tokens': 33, 'total_tokens': 55, 'input_token_details': {'audio': 0, 'cache_rea

In [ ]:
(prompt | llm ).invoke({"question": "In one sentence, how can LangSmith help with testing LLM apps?"})

AIMessage(content='LangSmith provides tools for creating, managing, and automating tests for LLM applications, enabling developers to ensure the accuracy and reliability of their models through structured testing frameworks.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 34, 'prompt_tokens': 33, 'total_tokens': 67, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_97c29773b5', 'id': 'chatcmpl-Db44O2dsdiqpKwsA9b5Qit1rKj1rO', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019de8bc-597b-7962-a38d-d1ac0aad9982-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 33, 'output_tokens': 34, 'total_tokens': 67, 'input_token_details': {'audio': 0,

## 3) RAG

We’ll load a small corpus (LangSmith docs), split it, embed it, index it with **FAISS**, and wire a **Retrieval Chain**.

- This section runs **once** and is reused later by the Agent.
- If web loading fails, we fall back to a tiny local sample so the demo still runs.
- We'll also **surface sources** so you can see why answers improved.

In [ ]:
# RAG (v1): Web loader → splitter → FAISS → retriever → LCEL chain
import os
os.environ.setdefault("USER_AGENT", "IK-LangChain-RAG/1.0 (contact: ops@your-org)")  # fixes the warning

from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1) Load docs (pick any public pages you want indexed)
urls = [
    "https://python.langchain.com/docs/get_started/introduction/",
    "https://docs.smith.langchain.com/"
]
loader = WebBaseLoader(urls)
docs = loader.load()

# 2) Chunk
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=200)
chunks = splitter.split_documents(docs)

# 3) Embed & index
emb = OpenAIEmbeddings()  # uses OPENAI_API_KEY from env # model = "text-embedding-ada-002"
vs = FAISS.from_documents(chunks, emb)
retriever = vs.as_retriever(search_kwargs={"k": 4})

# 4) Prompt (stuff-style: we inject all retrieved chunks into {context})
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a precise assistant. Use the provided CONTEXT to answer.\n"
     "If the answer isn't in the context, say you don't know.\n\nCONTEXT:\n{context}"),
    ("human", "{question}")
])

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)
    # return "\n-------------------------\n".join(d.page_content for d in docs)

# 5) LCEL pipeline: {question} flows through; {context} is produced by retriever
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 6) Try it
rag_chain.invoke("What is LangSmith and how does it relate to LangChain?")


'LangSmith is a framework-agnostic platform designed for building, debugging, and deploying AI agents and LLM applications. It allows users to trace requests, evaluate outputs, test prompts, and manage deployments all in one place. LangSmith is related to LangChain as it provides enhanced capabilities for LangChain agents, offering a more integrated workflow that combines observability, evaluation, deployment, and platform setup.'

In [ ]:
q = "What is LangSmith and how does it relate to LangChain?"
print("\n\nretriever.invoke(q)\n",retriever.invoke(q))
print("\n\nRunnablePassthrough().invoke(q)\n",RunnablePassthrough().invoke(q))
print("\n\nprompt.invoke(input={'context':retriever.invoke(q), 'question':RunnablePassthrough().invoke(q)})\n",prompt.invoke(input={'context':retriever.invoke(q), 'question':RunnablePassthrough().invoke(q)}))



retriever.invoke(q)
 [Document(id='3e500409-121f-4e23-8784-56f63c2f9d51', metadata={'source': 'https://docs.smith.langchain.com/', 'title': 'LangSmith docs - Docs by LangChain', 'language': 'en'}, page_content='previewAdditional resourcesData & complianceFAQLangSmith statusLangSmith docsCopy pageCopy pageDocumentation IndexFetch the complete documentation index at: https://docs.langchain.com/llms.txtUse this file to discover all available pages before exploring further.LangSmith is a framework-agnostic platform for building, debugging, and deploying AI agents and LLM applications. Trace requests, evaluate outputs, test prompts, and manage deployments all in one place, with your agent stack.'), Document(id='800231aa-1562-4eaa-b044-6273ee298d0e', metadata={'source': 'https://docs.smith.langchain.com/', 'title': 'LangSmith docs - Docs by LangChain', 'language': 'en'}, page_content='LangSmith docs - Docs by LangChainSkip to main contentJoin us May 13th & May 14th at Interrupt, the Agent 

## 4) Agent + Tools (on top of the same RAG)

We turn our retriever into a **Tool** and create an **OpenAI Functions Agent**.  
Optionally, if a **Tavily** API key is present, we add a web search tool.

> **Why an Agent?** It can decide *when* to use retrieval vs. answer directly, and sequence multi-step reasoning.

In [ ]:
# Agent + Tools (LangChain v1) — Structured JSON output

import os, json
from typing import Any, Dict, List, Optional
from pprint import pprint

from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_core.tools import tool, create_retriever_tool
from langchain_core.messages import AIMessage, ToolMessage

# Optional: Tavily web search tool
try:
    from langchain_community.tools.tavily_search import TavilySearchResults
except Exception:
    TavilySearchResults = None

# ------------------ Define tools ------------------
tools: List[Any] = []

@tool
def add(a: float, b: float) -> float:
    """Add two numbers."""
    return a + b


# def calculator(expression: str) -> float:
#     """Perform calculations"""
#     return float(eval(expression))

tools.append(add)

# Add retriever tool if your RAG cell created `retriever`
if "retriever" in globals():
    kb_tool = create_retriever_tool(
        globals()["retriever"],
        name="kb_search",
        description="Search the indexed KB/curriculum docs and return relevant passages."
    )
    tools.append(kb_tool)

# Optional Tavily tool (requires TAVILY_API_KEY)
if TavilySearchResults and os.getenv("TAVILY_API_KEY"):
    tools.append(TavilySearchResults(max_results=5, include_answer=True))

# ------------------ Build agent ------------------
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=(
        "You are a helpful technical assistant. Use tools when useful. "
        "Prefer kb_search for questions about our course/KB; use web search for general web info."
    ),
)

# ------------------ Helpers for structured output ------------------
def _final_text(res: Any) -> str:
    """Return final assistant text from AIMessage or from a {messages:[...]} dict."""
    if isinstance(res, AIMessage):
        return res.content or ""
    if isinstance(res, dict) and "messages" in res:
        for m in reversed(res["messages"]):
            if isinstance(m, AIMessage) or getattr(m, "type", "") == "ai":
                return getattr(m, "content", "") or ""
    return str(res)

def _collect_tool_calls_and_outputs(res: Any, max_len: int = 500) -> List[Dict[str, Any]]:
    """Return [{'name','args','output'}...] by pairing tool calls to ToolMessage outputs."""
    messages: List[Any] = []
    if isinstance(res, dict) and "messages" in res:
        messages = res["messages"]

    # Find first AI message with tool_calls (some runtimes store it in .tool_calls, others in additional_kwargs)
    tool_calls: List[Dict[str, Any]] = []
    for m in messages:
        if isinstance(m, AIMessage) and getattr(m, "tool_calls", None):
            tool_calls = m.tool_calls or []
            break
        addkw = getattr(m, "additional_kwargs", {}) if hasattr(m, "additional_kwargs") else {}
        if addkw.get("tool_calls"):
            tool_calls = addkw["tool_calls"]
            break

    # Map tool_call_id -> output from ToolMessage
    outputs: Dict[str, str] = {}
    for m in messages:
        if isinstance(m, ToolMessage):
            outputs[getattr(m, "tool_call_id", None)] = (getattr(m, "content", "") or "")

    def trunc(s: Optional[str]) -> str:
        if not s:
            return ""
        return s[:max_len] + ("…" if len(s) > max_len else "")

    structured: List[Dict[str, Any]] = []
    for c in tool_calls:
        structured.append({
            "name": c.get("name"),
            "args": c.get("args"),
            "output": trunc(outputs.get(c.get("id")))
        })
    return structured

def _usage(res: Any) -> Optional[Dict[str, Any]]:
    """Best-effort token usage extraction."""
    if isinstance(res, AIMessage):
        return getattr(res, "response_metadata", {}).get("token_usage")
    if isinstance(res, dict) and "messages" in res:
        for m in reversed(res["messages"]):
            meta = getattr(m, "response_metadata", {}) if hasattr(m, "response_metadata") else {}
            if meta.get("token_usage"):
                return meta["token_usage"]
    return None

def to_structured(res: Any) -> Dict[str, Any]:
    return {
        "answer": _final_text(res),
        "tools": _collect_tool_calls_and_outputs(res),
        "usage": _usage(res),
    }

# ------------------ Run and show structured JSON ------------------
query = "What is 41 + 1? Also, if I ask about LangGraph later, how would you use kb_search?"
res = agent.invoke({"messages": [{"role": "user", "content": query}]})
pprint(to_structured(res), width=100)


{'answer': 'The result of \\( 41 + 1 \\) is \\( 42 \\).\n'
           '\n'
           'If you ask about LangGraph later, I would use the `kb_search` function to look up '
           'relevant information in our knowledge base or curriculum documents. This would help me '
           'provide you with accurate and detailed information about LangGraph.',
 'tools': [{'args': {'a': 41, 'b': 1}, 'name': 'add', 'output': '42.0'},
           {'args': {'query': 'LangGraph'},
            'name': 'kb_search',
            'output': 'Quickstart - Docs by LangChainSkip to main contentJoin us May 13th & May '
                      '14th at Interrupt, the Agent Conference by LangChain. Buy tickets >Docs by '
                      'LangChain home pageOpen sourceSearch...⌘KAsk AIGitHubTry LangSmithTry '
                      'LangSmithSearch...NavigationGet startedQuickstartDeep '
                      'AgentsLangChainLangGraphIntegrationsLearnReferenceContributePythonOverviewGet '
                     

In [ ]:
res

{'messages': [HumanMessage(content='What is 41 + 1? Also, if I ask about LangGraph later, how would you use kb_search?', additional_kwargs={}, response_metadata={}, id='552e8896-b367-4a85-b11f-f1036aca0511'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 192, 'total_tokens': 241, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_3cdb07a9da', 'id': 'chatcmpl-Db44ZMNRDyCiWtBRHJoN9ixoD4ln1', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019de8bc-87b8-7453-8d5e-33001b8fe1dc-0', tool_calls=[{'name': 'add', 'args': {'a': 41, 'b': 1}, 'id': 'call_QzzzaEBTWCpeBadaHjfdO4Hi', 'type': 'tool_call'}, {'name': 'kb_search'

In [ ]:
# Run multiple queries through the v1 agent and print tidy results

from pprint import pprint
from langchain_core.messages import AIMessage
from datetime import datetime
datetime.now()
queries = [
    "List two LangSmith capabilities that support evaluation and how to use them.",
    "Where do the docs explain tracing? Summarize in 3 bullets.",
    "Who is the president of France?",
    "What is today's date?",
    f"Who is the president of France? Today it is {datetime.now()}",
]

def _final_text(res):
    if isinstance(res, AIMessage):
        return res.content or ""
    if isinstance(res, dict) and "messages" in res:
        for m in reversed(res["messages"]):
            if isinstance(m, AIMessage) or getattr(m, "type", "") == "ai":
                return getattr(m, "content", "") or ""
    return str(res)

for q in queries:
    print("\n" + "=" * 26)
    print("AGENT Q:", q)

    # v1 agents expect a messages list; fall back to raw string if needed
    try:
        res = agent.invoke({"messages": [{"role": "user", "content": q}]})
    except Exception:
        res = agent.invoke(q)

    # If you used Option 2 earlier, show structured JSON; else print final text
    if "to_structured" in globals():
        pprint(to_structured(res), width=100)
    else:
        print("\nAGENT A:\n", _final_text(res))



AGENT Q: List two LangSmith capabilities that support evaluation and how to use them.
{'answer': 'LangSmith offers several capabilities that support evaluation, two of which are:\n'
           '\n'
           '1. **Evaluation of Outputs**:\n'
           '   - **How to Use**: This capability allows you to measure and track the quality of '
           'your AI applications over time. You can evaluate the outputs generated by your models '
           'to ensure they are consistent and trustworthy. To use this feature, you would '
           'typically integrate evaluation metrics into your workflow, allowing you to assess the '
           'performance of your AI agents continuously.\n'
           '\n'
           '2. **Observability**:\n'
           '   - **How to Use**: This feature provides visibility into every step your application '
           'takes, which helps in debugging and improving reliability. By enabling observability, '
           'you can trace requests and monitor the pe

## 5) Tiny UI (Gradio)

A minimal chat interface that routes user messages to the agent.  
If Tavily is not available, the agent still works with the retriever tool.

In [ ]:

import gradio as gr
from langchain_core.messages import AIMessage

# -- helpers --
def _final_text(res):
    if isinstance(res, AIMessage):
        return res.content or ""
    if isinstance(res, dict) and "messages" in res:
        for m in reversed(res["messages"]):
            if isinstance(m, AIMessage) or getattr(m, "type", "") == "ai":
                return getattr(m, "content", "") or ""
    return str(res)

def _to_messages(history, message):
    # gr.ChatInterface history is List[Tuple[user, assistant]]
    msgs = []
    for u, a in history:
        if u: msgs.append({"role": "user", "content": u})
        if a: msgs.append({"role": "assistant", "content": a})
    msgs.append({"role": "user", "content": message})
    return msgs

def _ensure_agent():
    """Reuse global `agent` if defined; otherwise build a minimal one."""
    global agent
    try:
        agent  # already built in earlier cells
        return agent
    except NameError:
        from langchain_openai import ChatOpenAI
        from langchain.agents import create_agent
        from langchain_core.tools import tool

        @tool
        def add(a: float, b: float) -> float:
            "Add two numbers."
            return a + b

        llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
        agent = create_agent(model=llm, tools=[add], system_prompt="You are helpful.")
        return agent

# -- chat function used by Gradio --
def chat_fn(message, history):
    try:
        ag = _ensure_agent()
        msgs = _to_messages(history, message)
        res = ag.invoke({"messages": msgs})  # v1 call
        return _final_text(res)
    except Exception as e:
        return f"Error: {e}"

# -- UI --
try:
    demo.close()  # close a previous demo if re-running in the same kernel
except Exception:
    pass

with gr.Blocks() as demo:
    gr.Markdown("# LangChain Agent Chat")
    gr.Markdown("Ask about your KB (kb_search) or general queries. Web search only if TAVILY_API_KEY is set.")
    gr.ChatInterface(chat_fn)
    gr.Markdown("Tip: Try “Where are tracing docs?” or “Multiply 3.5 and 4.”")

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://df6847aac5959b756d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 6) Custom Tool (@tool) example

One simple tool is enough to demonstrate schema and descriptions.  
The agent can call this tool if it detects a matching need.

## 7) Custom tools

Define new tools with the `@tool` decorator. Rebuild an agent by passing the updated tools list to `create_agent`, then invoke.


In [ ]:
# Custom tool (LangChain v1) — add `multiply`, rebuild agent, return structured JSON

import os, json
from typing import Any, Dict, List, Optional

from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_core.tools import tool, create_retriever_tool
from langchain_core.messages import AIMessage, ToolMessage

# --- Define a custom tool ---
@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers a and b."""
    return a * b

# --- Compose tools2 (reuse prior tools if they exist) ---
tools2: List[Any] = []
if "tools" in globals():  # from earlier cell
    tools2.extend(globals()["tools"])
tools2.append(multiply)

# Add retriever as a tool if your RAG cell created `retriever` and it's not already added
if "retriever" in globals() and not any(getattr(t, "name", "") == "kb_search" for t in tools2):
    tools2.append(create_retriever_tool(
        retriever,
        name="kb_search",
        description="Search the indexed KB/curriculum docs and return relevant passages."
    ))

# Optional Tavily search tool
try:
    from langchain_community.tools.tavily_search import TavilySearchResults
    if os.getenv("TAVILY_API_KEY") and not any(getattr(t, "name", "") == "tavily_search_results_json" for t in tools2):
        tools2.append(TavilySearchResults(max_results=5, include_answer=True))
except Exception:
    pass

# --- Build an agent (v1) ---
try:
    llm  # defined earlier?
except NameError:
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

SYSTEM_PROMPT = (
    "You are a helpful technical assistant. Use tools when useful. "
    "Prefer kb_search for questions about our course/KB; use web search for general web info."
)
agent2 = create_agent(model=llm, tools=tools2, system_prompt=SYSTEM_PROMPT)

# --- Helpers to pretty-print structured JSON (answer + tools used + outputs) ---
def _final_text(res: Any) -> str:
    if isinstance(res, AIMessage):
        return res.content or ""
    if isinstance(res, dict) and "messages" in res:
        for m in reversed(res["messages"]):
            if isinstance(m, AIMessage) or getattr(m, "type", "") == "ai":
                return getattr(m, "content", "") or ""
    return str(res)

def _collect_tool_calls_and_outputs(res: Any, max_len: int = 500) -> List[Dict[str, Any]]:
    messages: List[Any] = res.get("messages", []) if isinstance(res, dict) else []

    # find tool calls
    tool_calls = []
    for m in messages:
        if isinstance(m, AIMessage) and getattr(m, "tool_calls", None):
            tool_calls = m.tool_calls or []
            break
        ak = getattr(m, "additional_kwargs", {}) if hasattr(m, "additional_kwargs") else {}
        if ak.get("tool_calls"):
            tool_calls = ak["tool_calls"]; break

    # map tool_call_id -> ToolMessage content
    outputs = {getattr(m, "tool_call_id", None): (getattr(m, "content", "") or "")
               for m in messages if isinstance(m, ToolMessage)}

    def trunc(s: Optional[str]) -> str:
        if not s: return ""
        return s[:max_len] + ("…" if len(s) > max_len else "")

    return [{"name": c.get("name"), "args": c.get("args"), "output": trunc(outputs.get(c.get("id")))}
            for c in tool_calls]

def _usage(res: Any) -> Optional[Dict[str, Any]]:
    if isinstance(res, AIMessage):
        return getattr(res, "response_metadata", {}).get("token_usage")
    if isinstance(res, dict) and "messages" in res:
        for m in reversed(res["messages"]):
            meta = getattr(m, "response_metadata", {}) if hasattr(m, "response_metadata") else {}
            if meta.get("token_usage"):
                return meta["token_usage"]
    return None

def to_structured(res: Any) -> Dict[str, Any]:
    return {"answer": _final_text(res), "tools": _collect_tool_calls_and_outputs(res), "usage": _usage(res)}

# --- Demo ---
q = "Multiply 3.5 by 4 and then list two LangSmith evaluation features."
res = agent2.invoke({"messages": [{"role": "user", "content": q}]})
print(json.dumps(to_structured(res), indent=2))


{
  "answer": "The result of multiplying 3.5 by 4 is **14.0**.\n\nTwo features of LangSmith evaluation are:\n\n1. **Observability**: Gain visibility into every step your application takes to debug faster and improve reliability.\n2. **Evaluation**: Measure and track quality over time to ensure your AI applications are consistent and trustworthy.",
  "tools": [
    {
      "name": "multiply",
      "args": {
        "a": 3.5,
        "b": 4
      },
      "output": "14.0"
    },
    {
      "name": "kb_search",
      "args": {
        "query": "LangSmith evaluation features"
      },
      "output": "previewAdditional resourcesData & complianceFAQLangSmith statusLangSmith docsCopy pageCopy pageDocumentation IndexFetch the complete documentation index at: https://docs.langchain.com/llms.txtUse this file to discover all available pages before exploring further.LangSmith is a framework-agnostic platform for building, debugging, and deploying AI agents and LLM applications. Trace requests, 

## 7) Wrap-up & Next steps

You built an end-to-end app:
- Baseline LLM → **Prompted chain** → **RAG** → **Agent with tools** → **(Optional) UI**
- Re-used the **same retriever** everywhere (built once).
- Optionally enabled **LangSmith** tracing for observability.

**Ideas to extend:**
- Swap FAISS for your vector DB of choice.  
- Add **validators** (output schemas) and **evaluation** suites.  
- Add domain-specific tools (databases, calculators, internal APIs).

> If you want to run the chat UI: uncomment `demo.launch()` in Section 5.